# 2. Production Reality — Optimized / Streaming Pipeline

**Agenda slot: 20–35 min**

Most production voice AI companies today still don't use end-to-end speech
models. They use separate STT + LLM + TTS — it gives better control, lower
cost, and more flexibility. The "it's slow" story from notebook 1 is only
true if you implement it naively (wait for each stage to fully finish).
Modern systems optimize every stage instead.

Full agenda: [`../voice-agent.md`](../voice-agent.md)

## What a production-grade modular pipeline looks like

```
User speaks
 ↓
Streaming STT (Deepgram, Gladia, Speechmatics, Google, AssemblyAI, OpenAI gpt-4o-transcribe)
 ↓
Partial transcripts every 100-300ms
 ↓
LLM starts reasoning before user finishes speaking
 ↓
Tool calls execute in parallel
 ↓
Streaming TTS (ElevenLabs, Cartesia, OpenAI TTS, Azure)
 ↓
Audio streamed back immediately
```

### The five key optimizations

**1. Streaming STT** — instead of waiting for the user to stop talking, the
STT service continuously emits partial transcripts:

```
200ms → "I want"
400ms → "I want to"
700ms → "I want to book"
1.0s  → "I want to book a"
1.2s  → "I want to book a flight"
```

The LLM can begin processing before the sentence is complete. Demoed below —
OpenAI's own `gpt-4o-transcribe` / `gpt-4o-mini-transcribe` models support
this directly over a plain HTTPS request (`stream=True`), no separate
provider or WebSocket session required. (This is different from `whisper-1`,
used in notebook 1, which is request/response only.)

**2. Incremental LLM generation** — demoed below.

**3. Streaming TTS** — demoed below.

**4. Parallel tool execution** — if the user asks "What's the weather in
Pune?", a well-designed system identifies the tool call quickly, starts the
API request, and prepares the response template while the call is in flight —
instead of waiting for a long response before even starting the lookup.

**5. Voice Activity Detection (VAD)** — determines when the user has likely
finished speaking, rather than waiting for a long silence. Shaves hundreds of
milliseconds off every turn.

In [ ]:
import sounddevice as sd
import soundfile as sf
from openai import OpenAI
from dotenv import load_dotenv
import time

load_dotenv()
client = OpenAI()

SYSTEM_PROMPT = (
    "You are a concise voice assistant. Always reply in English, regardless "
    "of what language the user spoke in. Speak naturally, never exceed 2 "
    "sentences, avoid markdown or lists, and ask at most one question back."
)


def play_audio(filename):
    audio, samplerate = sf.read(filename)
    sd.play(audio, samplerate)
    sd.wait()

### Demo: live streaming STT with turn detection (VAD)

This is the real production pattern, not a simulation of it: audio streams to
the model continuously over a WebSocket while you're still talking. Partial
transcript deltas print live, as you speak — not after you stop. A
server-side Voice Activity Detector (VAD) — the same mechanism Deepgram,
AssemblyAI, and the GPT Realtime API all use — decides when you've paused,
instead of waiting on a fixed timer.

If you don't actually say anything (just silence or background noise), VAD
never fires, no transcript is ever produced, and — deliberately — no LLM call
follows. Run the cell below, then speak a question out loud and pause when
you're done.

In [ ]:
import asyncio
import base64
import json
import websockets

REALTIME_WS_URL = "wss://api.openai.com/v1/realtime?model=gpt-realtime"
RT_SAMPLE_RATE = 24000


async def listen_and_transcribe(max_listen_seconds=15, language="en"):
    """Streams mic audio live and prints partial transcript deltas as you
    speak. Stops the instant server-side VAD detects a pause. Returns
    (transcript, heard_speech) — heard_speech stays False if VAD never
    detected any speech at all (silence-only), so the caller can skip the
    LLM call entirely.

    `language` pins the transcription model to a specific language (ISO-639-1,
    e.g. "en") instead of letting it auto-detect — without this, gpt-4o-transcribe
    will happily switch to whatever language it thinks it's hearing."""
    headers = {"Authorization": f"Bearer {client.api_key}"}
    transcript = ""
    heard_speech = False
    done = asyncio.Event()

    async with websockets.connect(REALTIME_WS_URL, additional_headers=headers, max_size=None) as ws:
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                "audio": {"input": {
                    "format": {"type": "audio/pcm", "rate": RT_SAMPLE_RATE},
                    "transcription": {"model": "gpt-4o-transcribe", "language": language},
                    "turn_detection": {
                        "type": "server_vad",
                        "silence_duration_ms": 500,
                        "create_response": False,
                    },
                }},
            },
        }))

        loop = asyncio.get_running_loop()
        audio_queue = asyncio.Queue()

        def on_audio(indata, frames, time_info, status):
            loop.call_soon_threadsafe(audio_queue.put_nowait, bytes(indata))

        mic_stream = sd.RawInputStream(
            samplerate=RT_SAMPLE_RATE, blocksize=int(RT_SAMPLE_RATE * 0.1),
            channels=1, dtype="int16", callback=on_audio,
        )

        async def sender():
            with mic_stream:
                print("Listening... speak your question, then pause.")
                while not done.is_set():
                    chunk = await audio_queue.get()
                    await ws.send(json.dumps({
                        "type": "input_audio_buffer.append",
                        "audio": base64.b64encode(chunk).decode("ascii"),
                    }))

        async def receiver():
            nonlocal transcript, heard_speech
            async for message in ws:
                event = json.loads(message)
                t = event["type"]
                if t == "input_audio_buffer.speech_started":
                    heard_speech = True
                elif t == "conversation.item.input_audio_transcription.delta":
                    transcript += event["delta"]
                    print(event["delta"], end="", flush=True)
                elif t == "conversation.item.input_audio_transcription.completed":
                    print("\n[pause detected — turn complete]")
                    done.set()
                    break

        sender_task = asyncio.create_task(sender())
        try:
            await asyncio.wait_for(receiver(), timeout=max_listen_seconds)
        except asyncio.TimeoutError:
            print("\n[no speech detected — a production system stops here, no LLM call]")
        finally:
            done.set()
            sender_task.cancel()

    return transcript.strip(), heard_speech

In [ ]:
t0 = time.time()
transcript, heard_speech = await listen_and_transcribe()
print(f"\nTotal listening time: {time.time() - t0:.1f}s")

if not heard_speech or not transcript:
    print("\nNo speech detected — skipping the LLM call entirely. A naive\n"
          "fixed-duration recording (notebook 1's approach) would have called\n"
          "Whisper, the LLM, and TTS regardless — even on pure silence.")
else:
    print(f"\nHeard: {transcript!r}")

In [ ]:
def stream_llm_answer(transcript):
    """Streams the LLM's reply token-by-token as it arrives. Returns the full answer text."""
    answer = ""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": transcript},
    ]
    for chunk in client.chat.completions.create(model="gpt-4o-mini", messages=messages, stream=True):
        delta = chunk.choices[0].delta.content
        if delta:
            answer += delta
            print(delta, end="", flush=True)
    print()
    return answer


def speak(answer, filename="stream_answer.wav"):
    """Streams the answer to speech and plays it back."""
    with client.audio.speech.with_streaming_response.create(model="tts-1", voice="alloy", input=answer, response_format="wav") as response:
        response.stream_to_file(filename)
    play_audio(filename)

### Demo: incremental LLM generation — only if we actually heard something

This is the "waits for pause, then generates incrementally" half of the
pattern: we only reach this cell with a real transcript once VAD confirmed
the turn is over. Feed it straight to the LLM and watch tokens print as they
arrive, instead of waiting for the full response — and instead of calling the
LLM at all if there was nothing to respond to.

In [ ]:
answer = None

if not transcript:
    print("No transcript — skipping the LLM call.")
else:
    t0 = time.time()
    answer = stream_llm_answer(transcript)
    print(f"\n(first-response time budget: {time.time() - t0:.2f}s)")

### Demo: streaming TTS

Audio bytes start arriving before the full response is synthesized. In a real
app you'd play chunks as they land (that's the overlap that makes production
systems feel instant); here we time it and buffer to a file for simplicity.

In [ ]:
if not answer:
    print("No answer to speak — skipping the TTS call too.")
else:
    t0 = time.time()
    speak(answer)
    print(f"Streaming TTS + playback finished in {time.time() - t0:.2f}s")

### Why this beats notebook 1's basic STT/TTS approach

Concrete, side-by-side:

| | Notebook 1 (naive) | This notebook (production-style) |
|---|---|---|
| Recording | Fixed 5s, always, no matter what you said | Streams continuously, ends the instant VAD detects a pause |
| Transcript feedback | Nothing until the whole recording finishes | Printed word-by-word while you're still talking |
| LLM call | Always made — even on silence | Only made if VAD actually heard speech |
| What happens on silence | `whisper-1` **hallucinates a phrase** on silence (we saw this ourselves — a Korean stock caption, "Thanks for watching"), the LLM answers it, TTS speaks a nonsense reply — 3 wasted API calls | Zero calls made. No cost, no latency, no nonsense answer |
| Wall-clock cost | Always ≈5s recording + full STT + full LLM + full TTS wait | ≈(however long you actually spoke) + a ~500ms pause + LLM/TTS latency — never longer than the conversation actually needs |

The mechanism that makes all of this possible is the same VAD-driven turn
detection GPT Realtime uses internally (notebook 3) — here we're using it
directly against the transcription-only side of the Realtime API, without
the full conversational session.

### Putting it together: the actual production loop

Everything above ran once, cell by cell, so each stage could be seen in
isolation. A real voice agent doesn't run STT once and stop — it **loops**:
listen, respond, listen again, for the entire conversation. That's what a
production server is actually doing the whole time a call is connected.

The loop below reuses the exact same building blocks from above
(`listen_and_transcribe`, `stream_llm_answer`, `speak`) — nothing new to
learn, just wired into a `for` loop over turns:

- Each turn starts by listening again.
- If VAD never heard speech that turn (silence, dead air, someone thinking),
  it skips straight back to listening — **no LLM call, no TTS call** — and
  moves to the next turn.
- Only a turn with real speech triggers the LLM → TTS chain.

Speak a question, pause, listen to the reply, then speak your next question
when it says "Listening..." again. Stay silent for one turn to see it skip
cleanly with no wasted calls.

In [ ]:
async def voice_loop(max_turns=3, listen_timeout=15):
    for turn in range(1, max_turns + 1):
        print(f"\n--- Turn {turn}/{max_turns} ---")
        transcript, heard_speech = await listen_and_transcribe(max_listen_seconds=listen_timeout)

        if not heard_speech or not transcript:
            print("No speech this turn — no LLM/TTS call, listening again.")
            continue

        print(f"You said: {transcript}\n")
        answer = stream_llm_answer(transcript)
        speak(answer)


await voice_loop(max_turns=3)

## Typical latency breakdown

A well-optimized modular pipeline might look like:

| Stage | Latency |
|---|---|
| Streaming STT | 150–300 ms |
| LLM first token | 150–400 ms |
| Streaming TTS first audio | 100–200 ms |

Users often hear the first spoken response in **400–900 ms** overall — a far
cry from notebook 1's multi-second wait.

## Then why use GPT Realtime at all?

Its value isn't just lower latency — it's **simplifying the architecture**.

With a modular stack, you manage: mic → STT provider → LLM → TTS provider →
audio player. That means handling different APIs, synchronization, partial
transcripts, audio formats, barge-in, conversation state, and timing between
every component yourself.

With GPT Realtime: mic → GPT Realtime → speaker. One session handles speech
recognition, language understanding, response generation, speech synthesis,
streaming, interruptions, turn detection, and conversation management. The
trade-off is **simplicity versus flexibility**.

## Why many companies still choose separate STT + LLM + TTS

Because they can optimize each component independently:

- Use the best STT for noisy call centers
- Use a domain-specific LLM for reasoning
- Choose a TTS voice that matches their brand
- Swap providers without redesigning the whole system
- Reduce costs by selecting cheaper services per stage

Common production stacks:

```
Deepgram → Claude / GPT / Llama → Cartesia
Gladia   → Gemini               → ElevenLabs
```

### Two valid architectures

**Architecture 1 — Modular Voice Stack** (most common in production)

| Pros | Cons |
|---|---|
| Lower operational cost in many cases | More engineering effort |
| Best-in-class components | Multiple APIs and failure points |
| Easier to customize | More orchestration required |
| Provider flexibility | |

**Architecture 2 — Unified Realtime Model**

| Pros | Cons |
|---|---|
| Much simpler architecture | Less control over individual components |
| Built-in interruption handling | Tied to a single provider's capabilities |
| Native multimodal interaction | May not be the most cost-effective at very high scale |
| Faster to prototype and launch | |

Both are legitimate choices in production today — which one wins depends on
scale, budget, and how much control you need over each stage.

**Next up:** `03_gpt_realtime.ipynb` — we build with the Unified Realtime
Model, and switch to the actual browser app for the full end-to-end demo.